# Trajectories of magnetic soft robot

### Imports

In [1]:
import numpy as np
import jax.numpy as jnp
import jax
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from softmobility import SoftBody, FlowBodySolver, shear_flow, oscillating_magnetic_field

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Numerical approach

### Parameter file

In [20]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - spring_k    
  - radius
  - length
input_names:
  - magnetic
    
# Default Values (Optional)
defaults:
  refradius: 1
  moment: 10
  spring_k: 10            
  radius: 0.5
  length: 0.5

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: refradius                    
    orientation: [0, 0, phi]
    torque: 
      - 0
      - 0
      - "moment * magnetic1 * cos(phi) - moment * magnetic0 * sin(phi) - spring_k * phi"              

  - # Sphere 1 #################
    radius: radius               
    position: [refradius + radius + length, 0, 0]       
    torque: [0, 0, spring_k * phi]    
"""

### Soft Body

In [21]:
mybody= SoftBody(yaml_data, verbose=True)

  Found variables: phi, length, radius, spring_k, magnetic0, magnetic1, magnetic2
  3D field inputs:  ['magnetic']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '0']
      Orientation: ['0', '0', 'phi']
      Coupling matrix C_H:
[['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['-10*sin(phi)', '10*cos(phi)', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['-spring_k']]
    Sphere 1
      Radius: radius
      Position: ['length + radius + 1', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['spring_k']]


### Flow-body solver

In [31]:
# Create a shear flow with shear rate 1
no_flow = shear_flow(shear_rate=0)
# Create a magnetic field 
magnetic_field = oscillating_magnetic_field(amp_x=1, amp_y=1, omega=1)

print(magnetic_field.vector(time=0))
print(magnetic_field.vector(time=0.2))

[ 1.  0.  0.]
[ 1.     0.199  0.   ]


In [32]:
# parameters
final_time = 12 * 2 * jnp.pi  
dt = 0.1

# optimal design parameters
mybody.set_design_defaults(new_dict={'spring_k': 18.103, 'radius': 0.511, 'length': 0.})

solver = FlowBodySolver(
    soft_body=mybody, 
    flow=no_flow, 
    input_map={"magnetic": magnetic_field}, 
    dt=dt, 
    init_orientation=[0, 0, 0], 
    integrator="RK2")
trajectory = solver.simulate(final_time=final_time)

OLD default param values: ['length', 'radius', 'spring_k'] [  0.      0.511  18.103]
NEW default param values: ['length', 'radius', 'spring_k'] [  0.      0.511  18.103]
Time: 0.000 / 75.398  Integrator RK2
Time: 10.000 / 75.398  Integrator RK2
Time: 20.000 / 75.398  Integrator RK2
Time: 30.000 / 75.398  Integrator RK2
Time: 40.000 / 75.398  Integrator RK2
Time: 50.000 / 75.398  Integrator RK2
Time: 60.000 / 75.398  Integrator RK2
Time: 70.000 / 75.398  Integrator RK2


### Figure of trajectory

In [33]:
# Extract position, orientation and dofs
positions = jnp.array([t[0] for t in trajectory])
orientations = jnp.array([t[1] for t in trajectory])
dofs = jnp.array([t[2][0] for t in trajectory])

# Time vector
t = jnp.linspace(0, final_time, len(trajectory))

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot DOFs in the first subplot
fig.add_trace(go.Scatter(x=t, y=dofs[:], mode='lines', name='DOF 0'), row=1, col=1)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=t, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=t, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

# Update layout for titles and labels
fig.update_layout(
    title="Trajectory Data Over Time",
    xaxis_title="Time (t)",
    # yaxis_title="Values (Position/Orientation Components)",
    template="plotly_white",  # Set the background to white
    showlegend=True,  # Show legend to distinguish between the traces
    height=800,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor='white',  # Set the plot background to white
    paper_bgcolor='white'  # Set the paper background to white
)

# Show the plot
fig.show()

## Theoretical approach, limit of small angles

### Parameter file

In [7]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - stiffness    
  - radius
  - length
input_names:
  - magnetic
    
# Default Values (Optional)
defaults:
  stiffness: 0.1            
  radius: 0.25
  length: 0.5
  stiffness_factor: 100
  radius_factor: 2
  length_factor: 2
  moment: 10
  refradius: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: refradius                    
    orientation: [0, 0, phi]
    torque: 
      - 0
      - 0
      - "moment * magnetic - stiffness_factor * stiffness * phi"              

  - # Sphere 1 #################
    radius: radius_factor * radius               
    position: [refradius + radius_factor * radius + length_factor * length, 0, 0]       
    torque: [0, 0, stiffness_factor * stiffness * phi]    
"""

### Extracting mobility components

In [8]:
theory_body= SoftBody(yaml_data, verbose=True)

  Found variables: phi, length, radius, stiffness, magnetic
  Scalar inputs:    ['magnetic']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '0']
      Orientation: ['0', '0', 'phi']
      Coupling matrix C_H:
[['0'], ['0'], ['0'], ['0'], ['0'], ['10']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['-100*stiffness']]
    Sphere 1
      Radius: 2*radius
      Position: ['2*length + 2*radius + 1', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[['0'], ['0'], ['0'], ['0'], ['0'], ['0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['100*stiffness']]


In [9]:
tensors = theory_body.compute_mobility_problem()
print("\nMobility Matrix M_H (multiplied by mu):")
print(tensors.M_H)
print("\nMobility Matrix M_K (multiplied by mu):")
print(tensors.M_K)


Mobility Matrix M_H (multiplied by mu):
[[ 0.   ]
 [ 0.001]
 [ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.023]
 [ 0.374]]

Mobility Matrix M_K (multiplied by mu):
[[ 0.   ]
 [-0.141]
 [ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.168]
 [-0.542]]


In [10]:
m1 = tensors.M_H[1,0]
m2 = tensors.M_H[-2,0]
m3 = tensors.M_H[-1,0]

k1 = tensors.M_K[1,0]
k2 = tensors.M_K[-2,0]
k3 = tensors.M_K[-1,0]

print("Magnetic coupling coefficients:", m1, m2, m3)
print("Elastic  coupling coefficients:", k1, k2, k3) 

Magnetic coupling coefficients: 0.00092426315 0.023460574 0.3739889
Elastic  coupling coefficients: -0.14099357 0.1676875 -0.5416764


In the limit of small angles we have
$$ \dot \theta_0 = -m_2 (\theta_0 + \phi) + m_2 B_y \sin(\omega t) + k_2 \phi, $$
$$ \dot \phi     = -m_3 (\theta_0 + \phi) + m_3 B_y \sin(\omega t) + k_3 \phi, $$

Calculating mean velocity in this limit with Wolfram calculation `Magnetic soft robot oneDOF`.

In [11]:
vmoy = ((k2*m1 - k1*m2)*m3)/(2*((1 + k3**2)*(1 + m2**2) - 2*(k2 + k3 - m2 + k2*k3*m2)*m3 + (1 + k2**2)*m3**2))
print("Mean velocity normalized by B_0y**2", vmoy)
print("Distance", vmoy * 10 * np.pi * 0.1**2)

Mean velocity normalized by B_0y**2 0.000372823
Distance 0.0001171258


### Optimization

In [12]:
def optimize_velocity(
    init_design,
    learning_rate=0.03,
    max_iters=1000,
    momentum = 0.9,  # Tune this (0.8 - 0.95 works well)
):
    def mean_v(design):
        dofs = jnp.zeros(theory_body.Ndof)
        tensors = theory_body.compute_mobility_problem(dofs, design)

        m1 = tensors.M_H[1,0]
        m2 = tensors.M_H[-2,0]
        m3 = tensors.M_H[-1,0]

        k1 = tensors.M_K[1,0]
        k2 = tensors.M_K[-2,0]
        k3 = tensors.M_K[-1,0]
        
        vmean = ((k2*m1 - k1*m2)*m3)/(2*((1 + k3**2)*(1 + m2**2) - 2*(k2 + k3 - m2 + k2*k3*m2)*m3 + (1 + k2**2)*m3**2))

        return vmean

    def constraints_design(design):
        return jnp.clip(design, 0, 1)

    design = init_design
    grad_mean_v = jax.jit(jax.grad(mean_v))
    velocity = jnp.zeros_like(design) 
        
    for i in range(max_iters):
        grad = grad_mean_v(design)
        grad_norm = jnp.linalg.norm(grad)

        if i % 100 == 0:
            print(i, mean_v(design), design, grad_norm)

        # Update velocity (exponential moving average of gradients)
        velocity = momentum * velocity + learning_rate * grad / (grad_norm + 1E-6)
        design = design + velocity

        # Enforce any additional constraints (like bounding design between 0 and 1)
        design = constraints_design(design)

    return design, mean_v(design)    

In [13]:
# Optimization trial
init_params = theory_body.design_defaults
opt_design, value = optimize_velocity(
    init_design=init_params,
    learning_rate=1E-4, 
    max_iters=3000,
)

0 0.000372823 [ 0.5   0.25  0.1 ] 0.0022676531
100 0.00053738256 [ 0.445  0.205  0.154] 0.0015568761
200 0.00068873435 [ 0.352  0.176  0.161] 0.0015990663
300 0.0008655898 [ 0.253  0.181  0.152] 0.0019417336
400 0.0010734616 [ 0.154  0.194  0.149] 0.002180116
500 0.0012872724 [ 0.057  0.218  0.154] 0.0019917511
600 0.0014044022 [ 0.     0.252  0.173] 0.0016223134
700 0.0014055909 [ 0.     0.255  0.18 ] 0.0017128453
800 0.0014056056 [ 0.     0.256  0.181] 0.0017249334
900 0.0014056044 [ 0.     0.256  0.181] 0.0017265441
1000 0.0014056051 [ 0.     0.256  0.181] 0.0017267675
1100 0.0014056063 [ 0.     0.256  0.181] 0.0017267975
1200 0.0014056063 [ 0.     0.256  0.181] 0.0017267975
1300 0.0014056063 [ 0.     0.256  0.181] 0.0017267975
1400 0.0014056063 [ 0.     0.256  0.181] 0.0017267975
1500 0.0014056063 [ 0.     0.256  0.181] 0.0017267975
1600 0.0014056063 [ 0.     0.256  0.181] 0.0017267975
1700 0.0014056063 [ 0.     0.256  0.181] 0.0017267975
1800 0.0014056063 [ 0.     0.256  0.181] 0.

In [14]:
print(theory_body.design_variables)
print(opt_design * np.array([2, 2, 100]))

['length', 'radius', 'stiffness']
[  0.      0.511  18.103]
